In [ ]:
import sys
sys.path.append("..")

import numpy as np
import joblib
import time
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from src.scoutai_common import get_engine, load_master_view, engineer_features, get_model, FULL_FEATURES, MODEL_PATHS

engine = get_engine()
df = load_master_view(engine)
df = engineer_features(df)

X = df[FULL_FEATURES]
y = df["log_current_market_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

old_pipeline = get_model("full")
preprocessor = old_pipeline.named_steps['preprocessor']
regressor = old_pipeline.named_steps['regressor']

old_preds = old_pipeline.predict(X_test)
old_rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(old_preds)))

In [ ]:
param_distributions = {
    'regressor__n_estimators': [400, 800, 1200, 1600],
    'regressor__max_depth': [4, 5, 6, 7, 8],
    'regressor__learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08],
    'regressor__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'regressor__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'regressor__min_child_weight': [1, 3, 5, 7],
    'regressor__gamma': [0, 0.1, 0.3, 0.5],
    'regressor__reg_alpha': [0, 0.1, 1],
    'regressor__reg_lambda': [1, 1.5, 2],
}

tuning_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', regressor)])

random_search = RandomizedSearchCV(
    estimator=tuning_pipeline, param_distributions=param_distributions, n_iter=25, cv=5,
    scoring='neg_mean_squared_error', random_state=42, n_jobs=1, verbose=1,
)

start_time = time.time()
random_search.fit(X_train, y_train)
elapsed_time = time.time() - start_time
print(f"[SYSTEM] Tuning completed in {elapsed_time:.1f} seconds!")

In [ ]:
best_pipeline = random_search.best_estimator_
new_preds = best_pipeline.predict(X_test)
new_rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(new_preds)))

print("Best Parameters Found:")
for key, value in random_search.best_params_.items():
    print(f"  * {key.replace('regressor__', '')}: {value}")
print(f"Old Model RMSE: E{old_rmse:,.0f}")
print(f"New Tuned RMSE: E{new_rmse:,.0f}")

if new_rmse < old_rmse:
    print("SUCCESS: tuned model is better -- overwriting scout_model_full.pkl")
    joblib.dump(best_pipeline, MODEL_PATHS["full"])
else:
    print("NOTE: old model performed at least as well -- keeping it.")